In [8]:
from keras.models import model_from_json

fichier_json = open('modele/modele.json', 'r')
modele_json = fichier_json.read()
fichier_json.close()

modele = model_from_json(modele_json)
modele.load_weights("modele/modele.weights.h5")

classes = [
    "Un T-shirt/haut", "Un pantalon", "Un pull", "Une robe", "Un manteau",
    "Des sandales", "Une chemise", "Des baskets", "Un sac", "Des bottes de cheville"
]

In [9]:
import numpy as np
from PIL import Image, ImageFilter

def load_and_preprocess_image(image_path, target_size=28, content_size=20):
    image = Image.open(image_path).convert('L')
    resized_img = resize_with_aspect_ratio(image, content_size)
    canvas = create_centered_image(resized_img, target_size)
    normalized_array = normalize_pixels(canvas)
    return normalized_array.reshape(1, target_size, target_size, 1)

def resize_with_aspect_ratio(image, target_dimension):
    width, height = image.size
    if width > height:
        new_width = target_dimension
        new_height = max(1, round(target_dimension * height / width))
    else:
        new_height = target_dimension
        new_width = max(1, round(target_dimension * width / height))
    return image.resize((new_width, new_height), Image.LANCZOS).filter(ImageFilter.SHARPEN)

def create_centered_image(image, canvas_size):
    canvas = Image.new('L', (canvas_size, canvas_size), 255)
    img_width, img_height = image.size
    x_offset = (canvas_size - img_width) // 2
    y_offset = (canvas_size - img_height) // 2
    canvas.paste(image, (x_offset, y_offset))
    return canvas

def normalize_pixels(image):
    pixels = np.array(image.getdata(), dtype=np.float32)
    return (255 - pixels) / 255.0

In [10]:
# Adapte le chemin si ton image est dans le sous-dossier "images/image.png"
image_array = load_and_preprocess_image("images/image.png")

probabilities = modele.predict(image_array)[0]
predicted_class = np.argmax(probabilities)

print(f"Selon moi l'image est : {classes[predicted_class]}\n")

for i, class_name in enumerate(classes):
    print(f"{class_name}: {probabilities[i] * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Selon moi l'image est : Une robe

Un T-shirt/haut: 3.19%
Un pantalon: 1.53%
Un pull: 0.15%
Une robe: 87.38%
Un manteau: 0.01%
Des sandales: 1.13%
Une chemise: 2.19%
Des baskets: 0.01%
Un sac: 4.40%
Des bottes de cheville: 0.01%


/tmp/ipykernel_40760/2359756187.py:30: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = np.array(image.getdata(), dtype=np.float32)
